# HPCQC Hands-on
## CPU, GPU and QPU backends with CUDA-Q

This notebook shows **how data moves between classical CPU, GPU, and QPU for a quantum kernel computation** on the LRZ HPCQC system. It is an infrastructure demo, not a quantum algorithm. The circuit is simply three rotations on qubits.

```
  host (CPU)                quantum backends
 ┌──────────────┐  sample  ┌──────────────────────────┐
 │              │ ───────▶ │ CPU backend              │  Case 1
 │              │ ◀─────── │ qpp-cpu (cudaq-sim)      │
 │              │  counts  └──────────────────────────┘
 │              │
 │              │  sample  ┌──────────────────────────┐
 │              │ ───────▶ │ GPU backend              │  Case 2
 │              │ ◀─────── │ nvidia (cudaq-sim)       │
 │              │  counts  └──────────────────────────┘
 │ list[float]  │
 │ counts       │  sample  ┌──────────────────────────┐
 │              │ ───────▶ │ multi-GPU backend        │  Case 3
 │              │ ◀─────── │ nvidia, mqpu (cudaq-sim) │
 │              │  counts  └──────────────────────────┘
 │              │
 │              │  sample  ┌──────────────────────────┐
 │              │ ───────▶ │ IQM QPU backend          │  Case 4
 │              │ ◀─────── │ iqm (LRZ HPC-QC)         │
 └──────────────┘  counts  └──────────────────────────┘
```

The host (CPU) always holds the parameters and the results. The four cases differ only in where the kernel runs, selected with `cudaq.set_target(...)`. Each run is timed.

**Run on BEAST via the portal:** use the `cudaq-gpu-env` kernel on a `wolpy` node. Case 3 needs two or more GPUs. Case 4 needs the `iqm_eqe1` license and an IQM token.

## Setup

In [6]:
import os
import time

import numpy as np
import cudaq
import matplotlib.pyplot as plt

SHOTS = 1000
N_SWEEP = 11                      # parameter sets in the sweep = circuits per run
SCALING_QUBITS = [10, 15, 20]     # circuit sizes for the CPU vs GPU comparison
MULTI_GPU_CIRCUITS = 8            # circuits in the case 3 batch

USE_QPU = False                   # True only in a session with the iqm_eqe1 license
IQM_SERVER_URL = os.environ.get("IQM_SERVER_URL", "https://qan06.beast.lrz.de/")
IQM_TOKEN_FILE = os.path.expanduser("~/.iqmclient_eqe1_token")

OUTPUT_DIR = "./hpcqc-output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_GPUS = cudaq.num_available_gpus()
HAS_GPU_SIM = N_GPUS > 0 and cudaq.has_target("nvidia")
cudaq.set_random_seed(11)

COLORS = {"CPU": "#2a78d6", "GPU": "#eb6834", "multi-GPU": "#1baf7a", "QPU": "#eda100"}
MARKERS = {"CPU": "o", "GPU": "s", "multi-GPU": "^", "QPU": "D"}

print(f"{cudaq.__version__.split('(')[0].strip()}  |  GPUs visible: {N_GPUS}")

CUDA-Q Version 0.14.2  |  GPUs visible: 0


## Kernels and helpers

`rotations` is the demo kernel. It applies `rz`, `ry` and `rx` with three angles from the host. Because `rz` acts on $|0\rangle$, it does not change the counts. The expected result is $P(1) = (1 - \cos a_1 \cos a_2)/2$, which we use to check every backend.

`layer` is a larger kernel for the timing comparisons. It has one qubit per angle, applies an `ry` to each qubit, and then a chain of CNOTs.

In [10]:
@cudaq.kernel
def rotations(a: list[float]):
    q = cudaq.qubit()
    rz(a[0], q)
    ry(a[1], q)
    rx(a[2], q)


@cudaq.kernel
def layer(theta: list[float]):
    q = cudaq.qvector(len(theta))
    for i in range(len(theta)):
        ry(theta[i], q[i])
    for i in range(len(theta) - 1):
        x.ctrl(q[i], q[i + 1])


print(cudaq.draw(rotations, [1.0, 2.0, 3.0]))

# Sweep on the host: a1 over [0, 2π], a0 and a2 fixed
a1 = np.linspace(0, 2 * np.pi, N_SWEEP)
ANGLES = [[1.0, x, np.pi / 4] for x in a1]
P1_EXACT = (1 - np.cos(a1) * np.cos(np.pi / 4)) / 2


def run_sweep():
    # Run the rotations sweep on the current target. Returns P(1) per point and total seconds.
    cudaq.sample(rotations, [0.0, 0.0, 0.0], shots_count=10)          # warm-up (compilation)
    t0 = time.perf_counter()
    p1 = np.array([cudaq.sample(rotations, a, shots_count=SHOTS).probability("1") for a in ANGLES])
    seconds = time.perf_counter() - t0
    print(f"{N_SWEEP} circuits in {seconds * 1e3:.1f} ms   max |P(1) - exact| = {np.abs(p1 - P1_EXACT).max():.3f}")
    return p1, seconds


def time_layer(n_qubits, repeats=3):
    # Average seconds per sample call of an n-qubit layer circuit on the current target.
    theta = np.random.default_rng(0).uniform(0, np.pi, n_qubits).tolist()
    cudaq.sample(layer, theta, shots_count=100)                       # warm-up
    t0 = time.perf_counter()
    for _ in range(repeats):
        cudaq.sample(layer, theta, shots_count=100)
    return (time.perf_counter() - t0) / repeats


def run_scaling():
    times = {}
    for n in SCALING_QUBITS:
        times[n] = time_layer(n)
        print(f"{n:3d} qubits: {times[n] * 1e3:9.1f} ms per sample call")
    return times


sweep = {}      # backend -> (P(1) array, seconds)
scaling = {}    # backend -> {qubits: seconds}

     ╭───────╮╭───────╮╭───────╮
q0 : ┤ rz(1) ├┤ ry(2) ├┤ rx(3) ├
     ╰───────╯╰───────╯╰───────╯



## Case 1: CPU only

The kernel runs on the CPU state-vector simulator `qpp-cpu`. This is the baseline.

In [11]:
cudaq.set_target("qpp-cpu")
sweep["CPU"] = run_sweep()
scaling["CPU"] = run_scaling()

11 circuits in 98.8 ms   max |P(1) - exact| = 0.028
 10 qubits:       9.9 ms per sample call
 15 qubits:      25.8 ms per sample call
 20 qubits:     816.4 ms per sample call


## Case 2: CPU + GPU

Same kernels, now on the GPU simulator `nvidia`. For one qubit, the GPU is no faster because launch overhead dominates. A simulator stores $2^n$ amplitudes, so the cost doubles with every qubit, and that is where the GPU pulls ahead.

In [ ]:
if HAS_GPU_SIM:
    cudaq.set_target("nvidia")
    sweep["GPU"] = run_sweep()
    scaling["GPU"] = run_scaling()
else:
    print("No GPU simulator available: skipping case 2.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
for name, times in scaling.items():
    ax.plot(list(times), [t * 1e3 for t in times.values()], color=COLORS[name], lw=2,
            marker=MARKERS[name], ms=7, markeredgecolor="white", markeredgewidth=1.5, label=name)
ax.set_yscale("log")
ax.set_xticks(SCALING_QUBITS)
ax.set_xlabel("qubits")
ax.set_ylabel("ms per sample call, log scale")
ax.set_title("Simulation time vs. circuit size", loc="left")
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/case2-cpu-vs-gpu.png", dpi=160, bbox_inches="tight")
plt.show()

## Case 3: CPU + multi-GPU

With `option="mqpu"`, each GPU acts as a separate QPU. `sample_async` sends each circuit to a GPU (`qpu_id`), and the circuits run in parallel. We run a batch of the largest `layer` circuits on one GPU, then on all GPUs.

This splits *many circuits* across GPUs. To split *one large circuit* across GPUs, use `option="mgpu"`, launched with `mpirun` from a batch job.

In [ ]:
def run_layer_batch(n_qpus):
    rng = np.random.default_rng(1)
    batch = [rng.uniform(0, np.pi, SCALING_QUBITS[-1]).tolist() for _ in range(MULTI_GPU_CIRCUITS)]
    for i in range(n_qpus):                                             # warm-up on every GPU
        cudaq.sample_async(layer, batch[0], shots_count=100, qpu_id=i).get()
    t0 = time.perf_counter()
    futures = [cudaq.sample_async(layer, theta, shots_count=100, qpu_id=i % n_qpus)
               for i, theta in enumerate(batch)]
    [f.get() for f in futures]
    return time.perf_counter() - t0


multi_gpu = {}
if HAS_GPU_SIM and N_GPUS > 1:
    cudaq.set_target("nvidia", option="mqpu")
    n_qpus = cudaq.get_target().num_qpus()
    for n in (1, n_qpus):
        multi_gpu[n] = run_layer_batch(n)
        print(f"{MULTI_GPU_CIRCUITS} x {SCALING_QUBITS[-1]}-qubit circuits on {n} GPU(s): {multi_gpu[n]:.2f} s")
    print(f"speedup: {multi_gpu[1] / multi_gpu[n_qpus]:.1f}x")
else:
    print(f"{N_GPUS} GPU(s) visible: case 3 needs at least 2. Skipping.")

## Case 4: CPU + QPU

The same `rotations` sweep runs on the IQM QPU. Each `sample` call is one job, so the time is mostly network and queueing, not the quantum run itself. Results also show hardware noise.

The IQM target reads the token from `IQM_TOKEN`. If that variable is not set, the notebook reads it from `~/.iqmclient_eqe1_token`.

In [ ]:
if USE_QPU:
    if not os.environ.get("IQM_TOKEN"):
        with open(IQM_TOKEN_FILE) as f:
            os.environ["IQM_TOKEN"] = f.read().strip()
    cudaq.set_target("iqm", url=IQM_SERVER_URL)
    sweep["QPU"] = run_sweep()
else:
    print("USE_QPU = False: skipping case 4.")

## Summary

The left plot shows that every backend returns the expected result, up to shot noise and, on the QPU, hardware noise. The right plot shows what the same sweep costs on each backend.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [3, 2]})

a1_fine = np.linspace(0, 2 * np.pi, 200)
ax1.plot(a1_fine, (1 - np.cos(a1_fine) * np.cos(np.pi / 4)) / 2, color="#52514e", lw=2, label="exact")
for name, (p1, _) in sweep.items():
    ax1.plot(a1, p1, linestyle="none", marker=MARKERS[name], ms=7, color=COLORS[name],
             markeredgecolor="white", markeredgewidth=1.5, label=name)
ax1.set_xlabel("a₁ (rad)")
ax1.set_ylabel("P(1)")
ax1.set_title(f"Results vs. exact ({SHOTS} shots)", loc="left")
ax1.legend(frameon=False, loc="lower center")

names = list(sweep)
ms = [sweep[n][1] * 1e3 for n in names]
ax2.barh(names, ms, color="#2a78d6", height=0.6)
for y, v in enumerate(ms):
    ax2.annotate(f"{v:,.0f} ms", (v, y), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)
ax2.invert_yaxis()
ax2.set_xscale("log")
ax2.set_xlim(right=max(ms) * 10)    # room for the value labels
ax2.set_xlabel("ms, log scale")
ax2.set_title(f"Time for {N_SWEEP} circuits", loc="left")

for ax in (ax1, ax2):
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/summary.png", dpi=160, bbox_inches="tight")
plt.show()

**Takeaways**

- The same kernel runs in all four cases. Only `cudaq.set_target` changes.
- **CPU:** fastest for small circuits.
- **GPU:** pays off as the qubit count grows.
- **Multi-GPU:** increases throughput when there are many circuits.
- **QPU:** time is mostly communication and queueing. That latency is what tighter GPU–QPU coupling such as NVQLink aims to reduce.